# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm==1.0.30 openpyxl thop scipy matplotlib scikit-learn

import sys, os, platform
from pathlib import Path
import numpy as np, pandas as pd, torch, timm

# Kaggle ghi vào /kaggle/working; Colab ghi vào /content.
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

REPO_DIR = str(WORK_DIR / "K4-Day02-NguyenDucTRieu-2A202602978")
if not Path(REPO_DIR).exists():
    !git clone https://github.com/ductrieunguyen897-code/K4-Day02-NguyenDucTRieu-2A202602978.git {REPO_DIR}

SUBMISSION_DIR = Path(REPO_DIR) / "submissions/2A202602978_NguyenDucTrieu"
CODE_DIR = SUBMISSION_DIR / "code"
# mkdir/cp starter trong hướng dẫn chỉ dùng khi bắt đầu làm bài.
# Bài này đã có code hoàn thiện: cần commit/push code lên GitHub trước khi clone.
# Không chép starter đè lên code đã làm.
assert (CODE_DIR / "experiments.py").is_file(), (
    "Repo clone chưa có code hoàn thiện. Hãy commit/push thư mục submissions lên GitHub; "
    "nếu đã clone trước đó, chạy git -C <REPO_DIR> pull --ff-only rồi chạy lại ô này."
)
sys.path.insert(0, REPO_DIR)                 # để import eval.py
sys.path.insert(0, str(CODE_DIR))             # mã bài làm, không import starter

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")


### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
IMAGES_DIR = "data/images"
LABELS_DIR = "data/labels"
assert Path(IMAGES_DIR).is_dir(), "Kiểm tra thư mục chứa ảnh sau giải nén ở ô trên."

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
from dataset import load_split, check_split
from prepare import eda
from IPython.display import display, Image

train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
split_stats = check_split(train_df, val_df, test_df, IMAGES_DIR)
print("Số ảnh mỗi tập:", split_stats["n"])
display(pd.DataFrame(split_stats["per_class"]).sort_index())

eda_stats = eda("data", "eda")
print("Tỉ lệ lớp lớn nhất / nhỏ nhất:", eda_stats["imbalance_ratio"])
print("Kích thước và kênh ảnh train:", eda_stats["train_image_shapes_modes"])
# Đối chiếu số đếm thực tế với Table 1; xem ít nhất 3 ảnh mỗi lớp.
for name in ["class_counts.png", "examples.png"]:
    display(Image(filename=f"eda/{name}"))

In [ ]:
from train import set_seed
from prepare import sanity

set_seed(0)
checks = sanity("data", "eda", backbone="resnet50")
print("CE khởi đầu:", checks["initial_ce"], "| ln(9):", np.log(9))
print("Loss sau overfit một batch:", checks["final_loss"])
assert checks["passed"] and checks["initial_ce_near_uniform"], "Dừng để kiểm tra pipeline."
display(Image(filename="eda/augmentation.png"))

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
from dataclasses import replace, asdict
from train import Config, run
from experiments import BACKBONES, base_config, result, dump
from report import generate

OUT = Path.cwd()  # /kaggle/working: nơi có quyền ghi
assert not (OUT / "final_lock.json").exists(), "Đã khóa chung kết; không chọn lại cấu hình."
base = base_config(OUT, "data", epochs=12, batch=64)
backbone_configs = []
for i, backbone in enumerate(BACKBONES, 1):
    cfg = replace(base, exp_id=f"B{i:02}", backbone=backbone, seed=0)
    run(cfg)
    backbone_configs.append(cfg)

best_backbone = max(backbone_configs, key=result)
baseline = replace(best_backbone, exp_id="T00")
dump(OUT / "selected_backbone.json", asdict(baseline))
generate(OUT)
display(pd.read_excel("results.xlsx", sheet_name="Backbones"))
print("Chọn", baseline.backbone, "vì có macro-F1 val cao nhất:", result(best_backbone))
# Sàng một seed: chưa kết luận khác biệt có ý nghĩa thống kê.

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
from experiments import ABLATIONS, read_cfg

assert not (OUT / "final_lock.json").exists(), "Đã khóa chung kết."
baseline = read_cfg(OUT / "selected_backbone.json")
run(baseline)
training_configs = [baseline]
# A: khởi tạo; B: augmentation; C: loss. Mỗi lượt chỉ đổi một yếu tố từ T00.
for exp_id, axis, changes in ABLATIONS:
    print(exp_id, "trục", axis, changes)
    cfg = replace(baseline, exp_id=exp_id, **changes)
    run(cfg)
    training_configs.append(cfg)

# T09: kết hợp lựa chọn tốt nhất trên val của từng trục.
combined_changes = {}
for axis in ["A", "B", "C"]:
    ids = {exp_id for exp_id, a, _ in ABLATIONS if a == axis}
    candidates = [baseline] + [cfg for cfg in training_configs if cfg.exp_id in ids]
    winner = max(candidates, key=result)
    if winner.exp_id != "T00":
        combined_changes.update(next(changes for exp_id, _, changes in ABLATIONS
                                     if exp_id == winner.exp_id))
combined = replace(baseline, exp_id="T09", **combined_changes)
run(combined)
training_configs.append(combined)
best_training = max(training_configs, key=result)
dump(OUT / "selected_training.json", asdict(best_training))
generate(OUT)
display(pd.read_excel("results.xlsx", sheet_name="Training"))
# Delta hiện là một seed; so với std sau khi chạy chung kết, không suy diễn từ delta nhỏ.

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
from experiments import METHODS, inference_stage

assert not (OUT / "final_lock.json").exists(), "Đã khóa chung kết."
# I00: 1-view; I01: hflip/prob; I02: hflip/logit; I03: five-crop; I04: temperature.
# Dùng checkpoint đã train; T chỉ fit trên val.
# Warmup 10, đo 100 lượt, synchronize CUDA; batch 1 và 32; p50/p95/p99.
print(METHODS)
inference_stage(OUT)
generate(OUT)
display(pd.read_excel("results.xlsx", sheet_name="Inference"))
display(pd.read_excel("results.xlsx", sheet_name="Latency"))
display(Image(filename="inference_tradeoff.png"))

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
from experiments import final_stage

# Hàm này gọi train.run cho F01 và T00 với cùng seed (0, 1, 2).
# Khóa lựa chọn từ val trước khi suy luận test; lưu đúng xác suất sau TTA/temperature.
# Test được quản lý riêng ở final_stage thay cho cờ save_test_predictions trong ví dụ mẫu.
# Dùng eval.save_predictions; nếu temperature được chọn, lưu thêm bản chưa hiệu chuẩn.
# Khi resume, dự đoán test đã hoàn thành được đọc lại, không chạy model lần nữa.
final_stage(OUT)

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
import shutil
import zipfile

# report.generate dùng pandas.ExcelWriter(engine="openpyxl"), chỉ đọc kết quả thật.
generate(OUT)
for sheet in ["Backbones", "Training", "Inference", "Final", "PerClass", "Latency", "Summary"]:
    print(sheet)
    display(pd.read_excel("results.xlsx", sheet_name=sheet))

# Kiểm tra mỗi thí nghiệm train có đường cong theo từng seed.
import json
for config_path in (OUT / "runs").glob("*/seed*/config.json"):
    cfg = json.loads(config_path.read_text())
    curve = OUT / "curves" / f"{cfg['exp_id']}_{cfg['backbone']}_seed{cfg['seed']}.png"
    assert curve.is_file(), f"Thiếu biểu đồ: {curve}"

# Gom đúng cấu trúc bài nộp; giữ checkpoint trong runs/ để resume, không đưa vào ZIP.
submission = OUT / "submission" / "2A202602978_NguyenDucTrieu"
submission.mkdir(parents=True, exist_ok=True)
for name in ["results.xlsx", "report.md", "grade.txt", "inference_tradeoff.png"]:
    if (OUT / name).exists(): shutil.copy2(OUT / name, submission / name)
for name in ["curves", "predictions", "eda", "analysis", "eval_out"]:
    if (OUT / name).exists(): shutil.copytree(OUT / name, submission / name, dirs_exist_ok=True)
shutil.copytree(CODE_DIR, submission / "code", dirs_exist_ok=True)
shutil.copy2(SUBMISSION_DIR / "README.md", submission / "README.md")
shutil.copy2(Path(REPO_DIR) / "eval.py", submission / "eval.py")
with zipfile.ZipFile(OUT / "submission_2A202602978.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for path in submission.rglob("*"):
        if path.is_file() and "__pycache__" not in path.parts:
            z.write(path, path.relative_to(submission.parent))
print("Bài nộp:", OUT / "submission_2A202602978.zip")
# Đọc report.md và ảnh lỗi để bổ sung phân tích cá nhân trước khi nộp.